In [1]:
# --- IMPORTS & DEVICE ---
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torchvision.models.resnet import resnet18
from torch.utils.data import DataLoader, Subset
from collections import OrderedDict
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors
import numpy as np
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
EMBEDDINGS_PATH = 'cifar10_simclr_embeddings.pt'

# initialise encoder
encoder = resnet18()
encoder.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
encoder.maxpool = nn.Identity()
encoder.fc = nn.Identity()
encoder.to(device)

Using device: cuda


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): Identity()
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), p

In [5]:
# run this script to get the embeddings saved from the .pth.tar file 


saved_weights_path = 'embeddings/model.pth.tar' 
checkpoint = torch.load(saved_weights_path, map_location=device)


state_dict = checkpoint.get('model', checkpoint)



new_state_dict = OrderedDict()
for k, v in state_dict.items():
    name = k[7:] if k.startswith('module.') else k
    name = name.replace('backbone.', '') 
    name = name.replace('shortcut.', 'downsample.')
    
    new_state_dict[name] = v


missing, unexpected = encoder.load_state_dict(new_state_dict, strict=False)
encoder.eval()

print(f"✅ Weights loaded!")
print(f"Missing keys from ResNet: {sum(1 for k in missing if 'encoder' in k)}")



clean_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2023, 0.1994, 0.2010])
])

clean_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=clean_transform
)

clean_loader = torch.utils.data.DataLoader(
    clean_dataset,
    batch_size=512,
    shuffle=False
)

# k means if k is less than 50 other minibatch k means


all_features = []
all_labels = []

with torch.no_grad():
    print("Features")

    for images, labels in clean_loader:
        images = images.to(device)
        
        features = encoder(images)

        all_features.append(features.cpu())
        all_labels.append(labels.cpu())


all_features = torch.cat(all_features, dim=0)
all_labels = torch.cat(all_labels, dim=0)

all_features_normalized = F.normalize(all_features, p=2, dim=1)

# --- Save the Embeddings ---
torch.save({
    'features': all_features_normalized,
    'labels': all_labels
}, 'cifar10_simclr_embeddings.pt')

torch.save(new_state_dict, 'cifar10_simclr_cleaned.pth')

print(f"Extraction complete! Final array shape: {all_features_normalized.shape}")

C:\Users\arnav\AppData\Local\Temp\ipykernel_13160\1875303703.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(saved_weights_path, map_location=dev

✅ Weights loaded!
Missing keys from ResNet: 0
Files already downloaded and verified
Features
Extraction complete! Final array shape: torch.Size([50000, 512])
